# What Are Physics-Informed Neural Networks?

**Time: ~20 minutes**

PINNs use neural networks to solve differential equations by embedding physics directly into the loss function. Instead of learning from labeled data, a PINN learns by satisfying the governing equations themselves.

## The Problem

Physics is described by differential equations — heat conduction, fluid flow, wave propagation, structural mechanics. Classical numerical solvers (FEM, FDM, spectral methods) have been the workhorses for decades. They work well, but they:

- Require **domain discretization** (meshing), which gets painful in complex geometries
- **Scale poorly** to high dimensions (curse of dimensionality)
- Need to be **re-run from scratch** for every new set of parameters
- Produce solutions that aren't easily **differentiable** for downstream optimization

What if a neural network could learn the solution function directly, without a mesh, by just knowing the equation?

## The Key Insight

A neural network is a **differentiable function**. Given input coordinates, it produces output values. Because it's built from differentiable operations, we can compute its derivatives exactly using automatic differentiation (autograd).

This is the entire idea:

```
Input (x, t)  →  Neural Network  →  Output u(x, t)
                       |
                    Autograd
                       |
                 du/dt, du/dx, d²u/dx²
                       |
             Residual = PDE(u, du/dt, d²u/dx², ...)
                       |
         Loss = mean(residual²) + mean(IC_error²) + mean(BC_error²)
```

We ask: "does this network's output satisfy the differential equation?" If the residual is large, the network is wrong. Minimize the residual → the network becomes the solution.

Three components of the loss:
- **PDE residual**: does the equation hold at collocation points in the domain?
- **Initial condition (IC) error**: does u match known values at t=0?
- **Boundary condition (BC) error**: does u satisfy constraints at domain edges?

In [ ]:
import torch
import torch.nn as nn

# A neural network is just a function approximator
model = nn.Sequential(
    nn.Linear(1, 32),
    nn.Tanh(),
    nn.Linear(32, 32),
    nn.Tanh(),
    nn.Linear(32, 1),
)

x = torch.linspace(0, 1, 5).view(-1, 1)
u = model(x)

print("Input x:", x.T)
print("Output u:", u.T.detach())
print("\nThe network maps coordinates → solution values. That's all it does.")

In [ ]:
# The magic: differentiating the network with autograd
x = torch.linspace(0, 1, 5).view(-1, 1).requires_grad_(True)
u = model(x)

du_dx = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]

print("u(x): ", u.T.detach())
print("du/dx:", du_dx.T.detach())
print("\nAutograd gives us EXACT derivatives of the network — not finite differences.")

In [ ]:
# Building a PDE residual: u' + u = 0
residual = du_dx + u  # For u' + u = 0, this should be zero
loss = torch.mean(residual**2)

print(f"Physics residual loss: {loss.item():.4f}")
print("It's NOT zero because the network is untrained — training minimizes this!")

## How Training Works

The training loop for a PINN:

1. **Sample collocation points** — random or structured points in the domain
2. **Forward pass** — feed points through the network to get u(x, t)
3. **Compute derivatives** — use autograd to get du/dt, du/dx, d²u/dx², etc.
4. **Evaluate the PDE residual** — plug everything into the equation
5. **Add IC/BC losses** — penalize violations of initial and boundary conditions
6. **Backpropagate** — compute gradients of total loss w.r.t. network weights
7. **Update weights** — optimizer step (Adam, L-BFGS, etc.)

After training, the network **is** the solution. You can evaluate it at any point in the domain instantly — no interpolation from a mesh, no re-solving.

## The Landscape: Types of PINNs

PINNs aren't a single technique — they're a family of approaches:

| Type | What it does | Example |
|------|-------------|--------|
| **Forward** | Solve a known PDE | Heat equation with given BCs |
| **Inverse** | Infer unknown parameters from data | Estimate viscosity from velocity measurements |
| **Parametric** | One model for a family of problems | Solve for any Reynolds number |
| **Data-augmented** | Combine physics + observations | Assimilate sensor data with governing equations |

The forward problem is the simplest and where we'll start, but inverse problems are where PINNs really earn their keep.

## When to Use PINNs (Honest Assessment)

**Good for:**
- **Inverse problems** — parameter identification from sparse, noisy data
- **Complex or irregular geometries** — mesh-free, no remeshing needed
- **High-dimensional PDEs** — no curse of dimensionality from meshing
- **Differentiable solutions** — optimization, control, sensitivity analysis
- **Multi-physics coupling** — embed multiple equations in one loss

**Bad for:**
- Well-posed problems where efficient classical solvers already exist
- Turbulence and chaotic dynamics
- Discontinuities and shocks (spectral bias)
- When you need guaranteed accuracy bounds or convergence certificates

**The rule:** if a classical solver works well for your problem, use it. PINNs shine where classical methods struggle — not as a blanket replacement.

## Common Misconceptions

**"PINNs replace FEM/FDM"** — No. They complement them. For most standard engineering problems, classical solvers are faster, more accurate, and better understood.

**"PINNs need training data"** — Not necessarily. A pure PINN is trained entirely from the equation and BC/IC — zero simulation data. Data can help, but isn't required.

**"PINNs are always better"** — They're often worse for simple, well-posed problems. Their advantage is flexibility, not raw performance.

**"One PINN = one solution"** — Parametric PINNs can solve entire families of problems. Parameters (BCs, material properties) become additional network inputs.

## What's Next

- **Notebook 02**: Deep dive into automatic differentiation — the engine that makes PINNs possible
- **Notebook 03**: Build a complete PINN from scratch, training it to solve a real ODE

The code cells above showed the three core pieces: a network, autograd derivatives, and a PDE residual. Everything else is engineering around these ideas.